# Hướng 2, bước 1: bản sao 292p của **tập train** (chỉ cần CPU)

Giống `kaggle_reencode_ek100.ipynb` (resize đúng như eval: cv2 `INTER_LINEAR`, cạnh ngắn 292, giữ mọi frame và đúng fps,
kiểm tra số frame/fps trong `manifest.csv`), với 2 khác biệt:

- **Nguồn:** `SOURCE = "hf"` lấy video gốc từ bản sao trên Hugging Face (`awsaf49/epic_kitchens_100`). Bản sao này chỉ có
  **video phần mở rộng EK100** (P01_101…: 201 video train, 33.6 giờ, **55.7% số segment train**), nên đặt `VIDEO_KIND = "extension"`.
  Mỗi file tải về vẫn được đối chiếu với `EPIC_100_video_info.csv` (fps, thời lượng, độ phân giải). Khi data.bris.ac.uk hoạt động
  lại: `VIDEO_KIND = "ek55"`, `SOURCE = "bristol"` cho 294 video còn lại (41 giờ).
- **Nén:** `yuv420p` crf 18 thay vì `yuv444p` crf 12 (bản val), để cả tập train vừa quota dataset (~0.8 GB mỗi giờ video, tức khoảng
  26 GB cho phần mở rộng, thay vì ~60 GB). Đo trên 2 video train (P03_110, P04_107): bản 292p crf 18 khớp video gốc ở mức tương đương
  bản val crf 12 (tập top-5 action trùng 45% so với 43%). Muốn đồng nhất tuyệt đối với bản val thì đặt `CRF = 12`, `PIX_FMT = "yuv444p"`.

**Cài đặt:** Accelerator **None**, Internet **On**. Mỗi phiên một `SHARD_ID` (0 … `NUM_SHARDS`−1); chạy song song được trên nhiều
tài khoản. Phiên dừng vì hết giờ / hết 19 GB: *Add Input* output của phiên đó rồi chạy lại cùng `SHARD_ID`. Xong thì tạo **Dataset từ
Output**, ví dụ `ek100-292-train-ext-0`.

In [ ]:
SHARD_ID = 0
NUM_SHARDS = 3            # extension: 3 (khoảng 9 GB và vài giờ mỗi shard); ek55: 3
VIDEO_KIND = "extension"  # "extension" (có trên Hugging Face) | "ek55" (chỉ có trên Bristol) | "all"
SOURCE = "hf"             # "hf" | "bristol" | "auto" (Hugging Face trước, rồi Bristol)
CRF, PIX_FMT = 18, "yuv420p"

In [ ]:
BRANCH = "direction2-balanced-head"
!git clone -q https://github.com/1Tyson/Testing-And-Research-VJEPA.git -b $BRANCH /tmp/repo
!apt-get -qq install -y aria2 ffmpeg > /dev/null 2>&1; pip install -q decord av imageio-ffmpeg
R = "/tmp/repo"
!git clone -q --depth 1 https://github.com/epic-kitchens/epic-kitchens-100-annotations.git /tmp/ek100-ann
!nproc; df -h /kaggle/tmp /kaggle/working | tail -2

In [ ]:
# Resume: copy the output of an earlier run of THIS shard (attach it via Add Input) so the new output is complete
import glob, os, shutil
OUT = "/kaggle/working/ek100_292"
prev = [os.path.dirname(m) for m in glob.glob("/kaggle/input/**/ek100_292/manifest.csv", recursive=True)]
for d in prev:
    print("copying", d)
    shutil.copytree(d, OUT, dirs_exist_ok=True)
print("resumed from", len(prev), "earlier output(s)" if prev else "(fresh run)")

In [ ]:
!python $R/scripts/reencode_ek100.py --split train --video_kind $VIDEO_KIND --source $SOURCE \
    --shard_id $SHARD_ID --num_shards $NUM_SHARDS --crf $CRF --pix_fmt $PIX_FMT \
    --annotations_dir /tmp/ek100-ann --out_dir /kaggle/working/ek100_292 --tmp_dir /kaggle/tmp/raw \
    --encode_workers 2 --ffmpeg_threads 2 --parallel_downloads 2 --max_hours 11 --max_output_gb 19

In [ ]:
import pandas as pd
m = pd.read_csv("/kaggle/working/ek100_292/manifest.csv", dtype=str)
ok = m.ok == "True"
print(f"{ok.sum()} ok, {(~ok).sum()} failed")
print("median bitrate (Mbit/s):", pd.to_numeric(m.mbps[ok], errors="coerce").median())
filled = pd.to_numeric(m.filled_frames, errors="coerce").fillna(0)
print("videos with frames filled in (corrupt packets in the original):\n", m.loc[filled > 0, ["video_id", "filled_frames"]])
print(m.loc[~ok, ["video_id", "error"]])
!du -sh /kaggle/working/ek100_292